# 02_01 Tokens and vocabulary

*📖 Book: sections 2.2 and 2.3*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → [numbers] → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **numbers**
>
> Project files this connects to:
>
> - `src/homer_ai/tokenizer.py`
>
> **This notebook:** We need a **fixed** way to turn each character into a number and back: if `π` were 41 today and 12 tomorrow, a trained model would read nonsense.
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How to cut text into one-character tokens
- How to build a vocabulary, and give each token a number
- How to encode text into numbers, and decode numbers back into text
- How our real `CharTokenizer` does all of this on Book 1

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **encode / decode** | text → IDs, and IDs → text | `πολα` → `[4, 3, 2, 1]` |
| **round trip** | encode, then decode; you should get back exactly what you started with | `πολα` → numbers → `πολα` |

Words you've met before: **token** ([02_00](../02_text_to_numbers/02_00_chapter_overview.ipynb)), **vocabulary** ([02_00](../02_text_to_numbers/02_00_chapter_overview.ipynb)), **token ID** ([02_00](../02_text_to_numbers/02_00_chapter_overview.ipynb)).

## 3. The idea

### ❓ What's the simplest way to cut text into tokens?

Make every character a token. A **character** is any single symbol: a letter, a mark, a space. So yes, spaces and punctuation count too.

```
πολλα πολλα  →  π ο λ λ α ␣ π ο λ λ α
```

(␣ stands for a space.) It's simple, and every Greek word can be written this way.

### ❓ How do we decide which number each character gets?

1. Collect every different character in the text.
2. Put them in a fixed order (alphabetical).
3. Number them from 0.

That ordered list is the **vocabulary**, and a character's position in it is its token ID. Because we sort, the same text always gives the same numbers.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ How do I find all the different characters in a text?

A `set` keeps only one copy of each thing, so the set of a text holds each character once.

👉 **Predict first:** `"πολλα πολλα"` has 11 characters. How many will the set have?

In [ ]:
text = "πολλα πολλα"
set(text)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Five different characters: `π`, `ο`, `λ`, `α` and the space. A set has no fixed order, so yours may come out in a different order.

</details>

### ❓ Why sort them?

Because a set has no fixed order, and the numbering has to be the same every time. So we sort it.

👉 **Predict first:** which character will come first?

In [ ]:
chars = sorted(set(text))
chars

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The space comes first, because computers sort the space before the Greek characters. After that come the Greek characters, in alphabetical order.

</details>

### ❓ How do I give each character a number?

With a lookup table (a dictionary) from character to number. The first character gets 0, the next gets 1, and so on.

👉 **Predict first:** what number will `π` get?

In [ ]:
char_to_id = {ch: i for i, ch in enumerate(chars)}
char_to_id

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Each character now has a number, from 0 to 4. So `π` got 4.

</details>

### ❓ How do I go back from numbers to characters?

Build the same dictionary the other way round, number → character.

In [ ]:
id_to_char = {i: ch for i, ch in enumerate(chars)}
id_to_char

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The same pairs, flipped. Now we can look things up in both directions.

</details>

### ❓ So how do I encode and decode a word?

To encode, look up each character's number. To decode, look up each number's character and join them back into text.

👉 **Predict first:** what numbers will `πολα` become, and will decoding give back `πολα`?

In [ ]:
ids = [char_to_id[c] for c in "πολα"]
print(ids)
print("".join(id_to_char[i] for i in ids))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Four numbers, `[4, 3, 2, 1]`, and the same word back. That's a successful **round trip**.

</details>

## 5. Full example

The project has a real tokenizer, `CharTokenizer`, in `src/homer_ai/tokenizer.py`. It does exactly the steps above, plus the special tokens from the next notebook. Here it is on all of Book 1.

👉 **Predict first:** Book 1 has 135 different characters. Our vocabulary also has 2 special tokens. How big is the vocabulary?

In [ ]:
from homer_ai.tokenizer import CharTokenizer
from homer_ai.train_homer import BOOKS_DIR, load_books

book1 = load_books(BOOKS_DIR)[0]
tokenizer = CharTokenizer(book1)

print("characters in Book 1:", len(book1))
print("vocabulary size:", tokenizer.vocab_size)

first_line = book1.splitlines()[0]
ids = tokenizer.encode(first_line)
print(first_line)
print(ids[:10], "...")
print("round trip ok:", tokenizer.decode(tokenizer.encode(book1)) == book1)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

So, 19,701 characters and a vocabulary of 137 (135 characters plus the 2 special tokens). You also get the first line's numbers, and a perfect round trip of the whole book.

</details>

### ❓ What numbers does `μοῦσα` get in the real vocabulary?

Same idea, real size. `μοῦσα` ("Muse", in the first line) becomes one number per character:

```
μ    ο    ῦ    σ    α
37   40   128  44   26
```

Notice that `ο` is 40 here, and it's 40 in `πολλὰ` too. A character always gets the **same** number.

👉 **Predict first:** which number is the biggest, and why might `ῦ` be so far down the list?

In [ ]:
word = "μοῦσα"
ids = tokenizer.encode(word)
print("".join(f"{ch:<5}" for ch in word))
print("".join(f"{i:<5}" for i in ids))

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

`37 40 128 44 26`. The vocabulary is sorted, and characters with accents and breathing marks (like `ῦ`) sort after the plain letters. That's why they get the bigger numbers.

</details>

### ❓ Your turn: which numbers does `ἄνδρα` get?

`ἄνδρα` is the first word of the *Odyssey*, and you'll meet its five numbers again and again in the next chapters. Encode it with Book 1's real tokenizer.

👉 **Predict first:** `α` is ID 26. Will `ἄ`, the same letter with marks on it, be close to 26?

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
# 👉 do: turn the word "ἄνδρα" into its list of IDs, with the real tokenizer
andra_ids = None

In [ ]:
from homer_ai.lessons import check

check(
    "the IDs of ἄνδρα",
    andra_ids,
    [tokenizer.char_to_id[ch] for ch in "ἄνδρα"],
    coach="There should be 5 numbers, one for each character, from the real "
    "tokenizer: tokenizer.encode(...). The IDs of another word? Then check the "
    "spelling. ἄ carries two marks, so copy the word from the task.",
)

<details><summary>💡 Hint</summary>

The cell above did it for `μοῦσα`: `tokenizer.encode(word)`.

</details>

<details><summary>✅ Answer</summary>

```python
andra_ids = tokenizer.encode("ἄνδρα")
```

`[58, 38, 29, 42, 26]`. `ἄ` is 58 and plain `α` is 26: to the tokenizer they're two different characters, as far apart as any other two. The sorting decides the numbers, and `ἄ` sorts long after the plain letters.

</details>

### ❓ Which characters does Book 1 use most?

Count every character, then draw the 15 most common. Each label shows the character and its ID. The space and the line break are characters too, so they get a bar as well (`' '` and `'\n'`).

👉 **Predict first:** which will be the most common: a letter, the space, or the line break?

In [ ]:
from collections import Counter

import matplotlib.pyplot as plt

from homer_ai.studio_style import ACCENT, INK, MUTED, studio_axes, studio_figure

top = Counter(book1).most_common(15)
labels = [f"{ch!r}  ID {tokenizer.encode(ch)[0]}" for ch, _ in top]
counts = [n for _, n in top]

fig = studio_figure(
    "Book 1, character by character",
    f"The space leads with {counts[0]:,} of {len(book1):,} characters",
    size=(9, 7),
)
ax = studio_axes(fig, (0.2, 0.08, 0.72, 0.76))
ax.set_axisbelow(True)  # grid lines behind the bars
ax.set_xlim(0, counts[0] * 1.12)
bars = ax.barh(labels, counts, color=MUTED)
bars[0].set_color(ACCENT)
for bar, n in zip(bars, counts, strict=True):
    ax.text(
        n + 8, bar.get_y() + bar.get_height() / 2, f"{n:,}",
        va="center", color=INK, family="monospace", fontsize=10,
    )
ax.invert_yaxis()
ax.tick_params(axis="y", labelsize=11, labelcolor=INK)
ax.grid(axis="y", visible=False)
ax.set_xlabel("times it appears")
plt.show()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The space wins by a long way (2,731), then come `ν`, `ο` and `ε`. The line break (ID 0) makes the top 15 too. And notice the IDs have nothing to do with how common a character is. They just follow the sorted vocabulary.

</details>

### ❓ Greek has 24 letters. Why does Book 1 have 135 characters?

Because every accent, breathing mark and capital makes a different character. Plain `α` alone turns up in Book 1 in **10** forms: `ά α ἀ ἁ ἂ ἄ ἅ ὰ ᾶ ᾷ`. Add the capitals, the space, the punctuation marks and the line break, and you get to 135.

The line break (the end of each verse) is a character too. In code it's written `\n`. That's one character, even though you type it with two symbols.

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Why does the tokenizer have exactly `encode`, `decode` and `vocab_size`?

Because that's all the rest of the model needs from it. Any tokenizer with those three can be swapped in, a BPE one for example (02_03), without touching the model or the training code.

## 6. Recap

- Simplest tokens: one character each.
- Vocabulary = every different character, sorted, numbered from 0.
- Two dictionaries encode and decode; a round trip gives back exactly the same text.

**You can now:**

- build a vocabulary from any text: `sorted(set(text))`, numbered from 0
- encode a word into IDs and decode it back with the real `CharTokenizer`
- check a tokenizer with a round trip

**Next:** [02_02: Special tokens](02_02_special_tokens.ipynb): what about characters the vocabulary doesn't have?